# AdaLQO: Adaptive Learned Query Optimizer with Shifting Detector

In [1]:
import argparse
import time
import os

import numpy as np
import pandas as pd
import torch
import sys
import AdaLQO.utils as utils
from AdaLQO.utils import plot_res
from AdaLQO.utils import prediction
from AdaLQO.shift_detector import mmd,ws,ks_values_pca

sys.path.insert(0, 'bao_server')
import bao_server.model as bao_model
import copy
import random

# sys.path.append('ShiftHandler')
# from replay_buffer import summarizer

In [2]:
df = utils.load_data("tpc-ds")
BATCH_SIZE = 100
print(f"All data loaded: {df.shape}")

# Filter queries that use PostgreSQL's default execution timeout
df_psql_overtime = df[df.apply(lambda x: x['latency_list'][0] is None, axis=1)]
print(f"PSQL Timeout queries: {df_psql_overtime.shape}")

df = df[df.apply(lambda x: x['latency_list'][0] is not None, axis=1)]
# print(df.shape)
# Remove inf latencies
df["latency_list"] = df["latency_list"].apply(
    lambda xs: [x for x in xs if x is not None] if isinstance(xs, list) else []
)


All data loaded: (2567, 7)
PSQL Timeout queries: (228, 7)


In [3]:
mismatched_rows = df[df.apply(lambda x: len(x['latency_list']) != len(x['plans']), axis=1)]
# varify all latency/plan paired
assert(mismatched_rows.shape[0] == 0)

In [4]:
# Remove empty row (all plans timeout)
df = df[
    df["latency_list"].apply(lambda xs: isinstance(xs, list) and len(xs) > 0)
    & df["plans"].apply(lambda xs: isinstance(xs, list) and len(xs) > 0)
]
df = df.reset_index(drop=True)

print(df.shape)

(2339, 7)


In [5]:
data = utils.split_dataset(df, batch_size=BATCH_SIZE, random_state=None)

## Init Model


In [6]:
def train_bao_model(X,y):
    model = bao_model.BaoRegression(have_cache_data=False, verbose=False)
    model.fit_feature_extractor(X, y)
    model.fit_model(X, y, seed=42, ada_size=False)

    return model

def init_bao_model(train_data):
    X, y = utils.get_training_data(train_data)
    return X,y, train_bao_model(X, y)

def retrain_model(X, y, cur_data):
    X_cur, y_cur = utils.get_training_data(cur_data)

    new_X = X + X_cur
    new_y = y + y_cur

    return new_X, new_y, train_bao_model(new_X, new_y)

In [7]:
train_data = data[0][0]
# X,y,reg = init_bao_model(train_data)

## 3. Queries(Plans) Featurization

In [8]:
def embedding_plans(model, all_plans):
    trees = model._BaoRegression__tree_transform.transform(all_plans)
    return model._BaoRegression__net.get_fixed_features(trees)

def get_plans(queries):
    all_plans = []
    for plans in queries["plans"]:
        all_plans.extend(plans)
    return all_plans

## 4. Predict Other Group Data & Continual Learning

In [9]:
import bao_server.featurize as f

def safe_prediction(model, data):
    try:
        return prediction(model, data)
    except f.TreeBuilderError as e:
        print("TreeBuilderError during prediction:", e)
        print("Refitting feature extractor and model on current batch...")

        X = []
        y = []

        for _, row in data.iterrows():
            X.extend(row["plans"])
            y.extend(row["latency_list"])

        model.fit_feature_extractor(X, y)
        model.fit_model(X, y, seed=42, ada_size=False)

        return prediction(model, data)

In [10]:
def cl(dataset, shift_detect):
    res = []
    X, y, ori_reg = init_bao_model(dataset[0][0])
    base_embedding = embedding_plans(ori_reg, X)
    cur_reg = copy.deepcopy(ori_reg)
    retrain_counter = 0

    for i, phase in enumerate(data):
        for j, queries in enumerate(phase):
            if i+j == 0: continue
                # X,y, ori_reg = init_bao_model(queries)

            plans = get_plans(queries)
            cur_embedding = embedding_plans(cur_reg, plans)
            match shift_detect:
                case "mmd":
                    mmd_score = mmd(cur_embedding, base_embedding)
                    print(i, j, mmd_score)
                    # if detected shifting, retrain Bao Model with the last batch data
                    if mmd_score > 0.1:
                        X,y,cur_reg = retrain_model(X, y, queries)
                        base_embedding = embedding_plans(ori_reg, X)
                        retrain_counter += 1
                case "ks":
                    cur = cur_embedding.cpu().detach().numpy()
                    base = base_embedding.cpu().detach().numpy()
                    ks_score = ks_values_pca(cur, base)
                    print(i, j, ks_score)
                    # if detected shifting, retrain Bao Model with the last batch data
                    if ks_score.statistic > 0.2:
                        # retrain bao model
                        retrain_counter += 1
                case "ws":
                    cur = cur_embedding.cpu().detach().numpy()
                    base = base_embedding.cpu().detach().numpy()
                    ws_score = ws(cur, base)
                    print(i, j, ws_score)
                    # if detected shifting, retrain Bao Model with the last batch data
                    if ws_score > 0.5:
                        # retrain bao model
                        retrain_counter += 1
            try:
                ori_res = prediction(ori_reg, queries)
            except f.TreeBuilderError:
                ori_res = safe_prediction(ori_reg, queries)
            cur_res = safe_prediction(cur_reg, queries)
            cur_res["base_bao_latency"] = ori_res["bao_latency"]
            res.append(cur_res)
            pre_data = data[i][j]

    return res


## 5. Maximum Mean Discrepancy (MMD)

In [11]:
# res = cl(dataset=data, shift_detect="mmd")
# res_df = pd.concat(res,ignore_index=True)
# plot_res("shift detect", res_df, "results/tpcds/performance_10_mmd.png")

### Relationship between MMD score and Prediction performance

In [16]:
def analyze_mmd_regret_detailed(
    dataset,
    batch_out_path="results/tpcds/mmd_regret_batch.csv"
):
    batch_rows = []

    for i in range(len(dataset[0])):
        train_df = dataset[0][i]
        X_train, y_train, model = init_bao_model(train_df)
        base_embedding = embedding_plans(model, X_train)

        for j in range(len(dataset[0])):
            test_df = dataset[0][j]
            X_test = get_plans(test_df)
            cur_embedding = embedding_plans(model, X_test)
            mmd_score = mmd(cur_embedding, base_embedding)

            if hasattr(mmd_score, "detach"):
                mmd_value = float(mmd_score.detach().cpu().item())
            else:
                mmd_value = float(mmd_score)

            res = prediction(model, test_df)
            batch_rows.append({
                "train_batch": i,
                "test_batch": j,
                "mmd_score": mmd_value,
                "mean_regret": res["regret"].mean(),
                "median_regret": res["regret"].median(),
                "max_regret": res["regret"].max(),
                "num_queries": len(res),
                "same_batch": i == j
            })

            print(
                f"train_batch={i}, test_batch={j}, "
                f"mmd={mmd_value:.4f}, "
                f"mean_regret={res['regret'].mean():.4f}"
            )

    batch_df = pd.DataFrame(batch_rows)
    batch_df.to_csv(batch_out_path, index=False)

    return batch_df

In [17]:
analyze_mmd_regret_detailed(dataset=data)

train_batch=0, test_batch=0, mmd=0.0000, mean_regret=1.5051
train_batch=0, test_batch=1, mmd=0.0215, mean_regret=97.2903
train_batch=0, test_batch=2, mmd=0.0488, mean_regret=5.3438
train_batch=0, test_batch=3, mmd=0.0477, mean_regret=24.1735
train_batch=0, test_batch=4, mmd=0.0282, mean_regret=5.5175
train_batch=0, test_batch=5, mmd=0.0300, mean_regret=16.8058
train_batch=0, test_batch=6, mmd=0.0196, mean_regret=6.3371
train_batch=0, test_batch=7, mmd=0.0320, mean_regret=7.5524
train_batch=0, test_batch=8, mmd=0.0275, mean_regret=7.2567
train_batch=0, test_batch=9, mmd=0.0212, mean_regret=12.8035
train_batch=0, test_batch=10, mmd=0.0311, mean_regret=12.0048
train_batch=0, test_batch=11, mmd=0.0186, mean_regret=7.3006
train_batch=0, test_batch=12, mmd=0.0262, mean_regret=15.1305
train_batch=0, test_batch=13, mmd=0.0347, mean_regret=35.3104
train_batch=0, test_batch=14, mmd=0.0458, mean_regret=16.4782
train_batch=0, test_batch=15, mmd=0.0414, mean_regret=5.3958
train_batch=0, test_batch=

,train_batch,test_batch,mmd_score,mean_regret,median_regret,max_regret,num_queries,same_batch
0,0,0,0.000000,1.505142,1.193552,4.939607,100,True
1,0,1,0.021536,97.290300,1.292824,6733.836842,100,False
2,0,2,0.048824,5.343842,1.362410,143.028169,100,False
3,0,3,0.047685,24.173504,1.396848,1966.977273,100,False
4,0,4,0.028201,5.517474,1.229573,253.555556,100,False
...,...,...,...,...,...,...,...,...
571,23,19,0.039264,231.620639,1.252731,22986.911765,100,False
572,23,20,0.033801,6.221252,1.444837,201.209336,100,False
573,23,21,0.030415,3.896201,1.287663,139.957041,100,False
574,23,22,0.043220,138.095284,1.247118,5714.618421,100,False


## 7. Kolmogorov-Smirnov (KS) test

In [14]:
# reg = init_bao_model(data)
# res = cl(reg,"ks",data)
# res_df = pd.concat(res,ignore_index=True)
# plot_res("shift detect", res_df, "results/tpcds/performance_10_ks.png")

### KL-Divergence Or JSD vs Regrets (Optional)

## 9. Wassertein Distance vs Regrets

In [15]:
# reg = init_bao_model(data)
# res = cl(reg,"ws",data)
# res_df = pd.concat(res,ignore_index=True)
# plot_res("shift detect", res_df, "results/tpcds/performance_20_ws.png")

Therefore, I recommend using the following main figures for the final paper:

Normalized Latency (Default/Bao/Optimal) ← Main Figure

Regret vs Phase ← Core Results

MMD Score + Retrain Point ← Method Validation

Top-1 Accuracy ← Auxiliary Results

These four figures should be sufficient to fully support the experimental section of the entire Continual LQO paper.
因此我建议最终论文主图用：
Normalized Latency (Default/Bao/Optimal) ← 主图
Regret vs Phase ← 核心结果
MMD Score + Retrain Point ← 方法验证
Top-1 Accuracy ← 辅助结果
这四张图基本就能完整支撑整个 Continual LQO 论文的实验部分。